# Data inventory and index creation

This notebook validates the metadata and image files for one selected dataset and split.

It creates one row per image, and attaches the available jpeg and DICOM paths.


Intended outputs:
- 'subset_train_index.csv'
- 'subset_test_index.csv'
- 'full_train_index.csv'
- 'full_test_index.csv'

It does not load image-pixel data.

### Imports

In [93]:
from pathlib import Path
import pandas as pd
from PIL import Image

### Dataset Configuration

**Valid configurations**

- DATASET_SCOPE: 'subset', 'full'
- DATA_SPLIT: 'train', 'test'

In [94]:
DATASET_SCOPE = 'full'
DATA_SPLIT = 'test'

In [95]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "derived" / "indexes"

In [96]:
# conditional folder selection, depedent on the selection of dataset and data split
if DATASET_SCOPE == 'full':
    metadata_path = DATA_DIR / "metadata"/ f"{DATA_SPLIT}.csv"
    jpeg_dir = DATA_DIR / "jpeg" / DATA_SPLIT
    dicom_dir = DATA_DIR / "dicom" / DATA_SPLIT

else:
    metadata_path = DATA_DIR / DATASET_SCOPE / "metadata" / f"{DATA_SPLIT}.csv"
    jpeg_dir = DATA_DIR / DATASET_SCOPE / "jpeg" / DATA_SPLIT
    dicome_dir = None

In [84]:
metadata_df = pd.read_csv(metadata_path)

In [85]:
# create the jpeg path data frame

jpeg_files = list(jpeg_dir.glob("*.jpg"))

jpeg_index = pd.DataFrame({
    "image_name": [path.stem for path in jpeg_files],
    "jpeg_path": [path.relative_to(PROJECT_ROOT).as_posix() for path in jpeg_files]
                            })

In [86]:
combined_index = metadata_df.merge(jpeg_index, on='image_name', how='left',validate='one_to_one')

In [87]:
if dicom_dir is not None:
    dicom_files = list(dicom_dir.glob("*.dcm"))

    dicom_index = pd.DataFrame({
        "image_name": [path.stem for path in dicom_files],
        "dcm_path": [path.relative_to(PROJECT_ROOT).as_posix() for path in dicom_files]
                                })
    
    combined_index = combined_index.merge(dicom_index, on='image_name', how='left',validate='one_to_one')


    

In [88]:
combined_index['dataset_scope'] = DATASET_SCOPE
combined_index['data_split'] = DATA_SPLIT

In [89]:
# save csv file
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = (
    OUTPUT_DIR
    / f"{DATASET_SCOPE}_{DATA_SPLIT}_index.csv"
)

combined_index.to_csv(output_path, index=False)

In [90]:
print(output_path)

c:\Users\madis\Documents\ADS2002-Melanoma\data\derived\indexes\full_test_index.csv


### Summary
This process has been completed and created:
- 'subset_train_index.csv'
- 'subset_test_index.csv'
- 'full_train_index.csv'
- 'full_test_index.csv'

which reside in the data/derived/indexes folder

In [103]:
index_files = sorted(OUTPUT_DIR.glob("*_index.csv"))

index_summary = pd.DataFrame({"index": [path.name for path in index_files],
                            "rows": [len(pd.read_csv(path)) for path in index_files]})

In [104]:
index_summary

,index,rows
0,full_test_index.csv,10982
1,full_train_index.csv,33126
2,subset_test_index.csv,1988
3,subset_train_index.csv,6864
